In [87]:
import numpy as np
import pandas as pd


Reading the dataset

In [88]:
df = pd.read_csv('/kaggle/input/datasets/organizations/google/google-patent-phrase-similarity-dataset/train.csv')
df.head()

,anchor,target,rating,score,context
0,aralkynyl,cholesterol,1d,0.25,C09
1,aralkynyl,aralkyl,1d,0.25,C09
2,aralkynyl,heterocycle,1c,0.25,C09
3,aralkynyl,acyl,1d,0.25,C09
4,aralkynyl,heterocyclic,1c,0.25,C09


studying the dataset

In [89]:
df.describe(include= object)

,anchor,target,rating,context
count,36473,36473,36473,36473
unique,733,29340,10,106
top,component composite coating,composition,0,H01
freq,152,24,7471,2186


checking the null values

In [90]:
df.isna().sum()

anchor     0
target     0
rating     0
score      0
context    0
dtype: int64

checking the duplicate datas in df

In [91]:
df.duplicated().sum()

np.int64(0)

Creating a input feature, combining context, target and the anchor feature

In [92]:
df['input'] = 'TEXT1: ' + df['context'] + '; TEXT2: ' + df['target'] + '; ANC1: ' + df['anchor']

In [93]:
df.head()

,anchor,target,rating,score,context,input
0,aralkynyl,cholesterol,1d,0.25,C09,TEXT1: C09; TEXT2: cholesterol; ANC1: aralkynyl
1,aralkynyl,aralkyl,1d,0.25,C09,TEXT1: C09; TEXT2: aralkyl; ANC1: aralkynyl
2,aralkynyl,heterocycle,1c,0.25,C09,TEXT1: C09; TEXT2: heterocycle; ANC1: aralkynyl
3,aralkynyl,acyl,1d,0.25,C09,TEXT1: C09; TEXT2: acyl; ANC1: aralkynyl
4,aralkynyl,heterocyclic,1c,0.25,C09,TEXT1: C09; TEXT2: heterocyclic; ANC1: aralkynyl


Making a huggingface dataset from ourdataframe

In [ ]:
from datasets import Dataset,DatasetDict
ds = Dataset.from_pandas(df)    #converting a dataframe into a dataset         
ds

Dataset({
    features: ['anchor', 'target', 'rating', 'score', 'context', 'input'],
    num_rows: 36473
})

Taking a pre-trained model


In [95]:
model_nm = 'microsoft/deberta-v3-small'

Creation of tokenizer

In [96]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
tokz = AutoTokenizer.from_pretrained(model_nm)  #creating a tokenizer based on the pretrained model
tokz

DebertaV2Tokenizer(name_or_path='microsoft/deberta-v3-small', vocab_size=128000, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[CLS]', 'eos_token': '[SEP]', 'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	128000: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [97]:
def tokenize_data(x):
    return tokz(x['input'])  #tokenizing the input feature of a dataframe

Tokenization

In [98]:
tok_ds = ds.map(tokenize_data,batched=True)  #applying the tokenize function to each rows but in chunk, using batched = True 

Map:   0%|          | 0/36473 [00:00<?, ? examples/s]

In [99]:
first_row = tok_ds[0]
first_input = first_row['input']
first_ids = first_row['input_ids']
print('The first input data is:', first_input)
print('The first corresponding id is:', first_ids)

The first input data is: TEXT1: C09; TEXT2: cholesterol; ANC1: aralkynyl
The first corresponding id is: [54453, 435, 294, 716, 4505, 346, 54453, 445, 294, 9888, 346, 23702, 435, 294, 266, 17226, 9593, 63791]


In [100]:
tokz.vocab['cholesterol']

84044

As the huggingface transformers expect the labels to be named actually labels, so we are renaming the feautre score with labels.

In [101]:
tok_ds = tok_ds.rename_columns({'score':'labels'})

Splitting the dataset into train_test using train-test split

In [102]:
dds = tok_ds.train_test_split(0.20,seed = 42)    #we are using 20% of the total dataset as the test dataset
dds

DatasetDict({
    train: Dataset({
        features: ['anchor', 'target', 'rating', 'labels', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 29178
    })
    test: Dataset({
        features: ['anchor', 'target', 'rating', 'labels', 'context', 'input', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 7295
    })
})

Training the model


Correlation as a evaluation metrics

In [103]:
def corr(x,y):
    return np.corrcoef(x,y)[0][1]  #here we are extracting the correlation value between x and y from the correlation coefficient matrix
#transformer expects the metrics to be returned as a dictionary, so that the trainer knows what label to use
def corr_d(eval_pred): 
    return {'pearson': corr(*eval_pred)}

assigning the batchsize and number of epochs

In [104]:
bs = 128
epochs = 4
lr = 0.01

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding 

In [110]:
args = TrainingArguments('outputs', learning_rate=lr, warmup_steps=0.1, lr_scheduler_type='cosine', bf16=True,
    eval_strategy="epoch", per_device_train_batch_size=bs, per_device_eval_batch_size=bs*2,
    num_train_epochs=epochs, weight_decay=0.01, report_to='none')

In [112]:
data_collator = DataCollatorWithPadding(tokenizer = tokz)  #inorder to create the necessary padding around the tensors

In [113]:
model = AutoModelForSequenceClassification.from_pretrained(model_nm, num_labels = 1)
trainer = Trainer(model,args , train_dataset = dds['train'], eval_dataset = dds['test'], data_collator = data_collator, compute_metrics = corr_d)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight       

Training the model

In [114]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Pearson
1,No log,nan,nan
2,No log,nan,nan
3,No log,nan,nan
4,No log,nan,nan


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=456, training_loss=6.701921429550438, metrics={'train_runtime': 114.6829, 'train_samples_per_second': 1017.693, 'train_steps_per_second': 3.976, 'total_flos': 740289114047580.0, 'train_loss': 6.701921429550438, 'epoch': 4.0})